# Saving Analytical Queries as Views

**Assignment:** Milestone 5 – Save some of the queries as views
**Target Host:** 10.100.71.21
**Target Database / Schema:** 202618052 / DBMS_project
**Source Data:** Formula 1 World Championship (1950–2024)


## Objective

1. Writing the `CREATE VIEW` statements for ten of the Milestone 4 queries
2. Creating the views in the `DBMS_project` schema on the PostgreSQL server
3. Reading each view back with `read_sql_query` and printing `head(10)`
4. Listing all the views saved in the schema

## Views

Ten of the Milestone 4 analytical queries are saved as views in the `DBMS_project` schema. A view stores the query
under a name, so its result can later be read with a simple `SELECT` instead of rewriting the full query.
`LIMIT` clauses are removed so that each view returns the complete result.

| No. | View | Based on | Description |
|---|---|---|---|
| 1 | `v_driver_career_wins` | Query 1 | Career wins, podiums and win percentage of every driver |
| 2 | `v_constructor_championships` | Query 2 | Constructors' Championship titles and title years of every constructor |
| 3 | `v_driver_career_points` | Query 3 | Career championship points, seasons and titles of every driver |
| 4 | `v_pitstop_efficiency` | Query 4 | Pit-stop performance of each constructor across seasons |
| 5 | `v_dnf_analysis` | Query 9 | Race retirements of each constructor grouped by cause |
| 6 | `v_season_points_progression_2021` | Query 10 | Race-by-race championship progression of the 2021 top 5 |
| 7 | `v_home_race_advantage` | Query 12 | Home Grand Prix finishing position compared with the driver's other races |
| 8 | `v_decade_team_dominance` | Query 13 | Top 3 constructors of each decade by share of points |
| 9 | `v_grid_vs_points` | Query 14 | Points, wins and podiums by starting grid position (2010 onward) |
| 10 | `v_sprint_vs_gp` | Query 15 | Sprint result compared with the Grand Prix result on the same weekend |

1. `v_driver_career_wins`
2. `v_constructor_championships`
3. `v_driver_career_points`
4. `v_pitstop_efficiency`
5. `v_dnf_analysis`
6. `v_season_points_progression_2021`
7. `v_home_race_advantage`
8. `v_decade_team_dominance`
9. `v_grid_vs_points`
10. `v_sprint_vs_gp`

## Note on the uploaded data

The data was uploaded in Milestone 3 with `to_sql` exactly as it appears in the preprocessed CSV files.
In the `results` table, the columns `position` and `rank` therefore contain the text `\N` for missing values,
and `points` is stored as a decimal (double precision) number. The queries handle this directly in SQL:

- `NULLIF(r.position, '\N')::int` turns `\N` into `NULL` and the rest into whole numbers,
- `r.rank = '1'` selects the fastest lap of a race,
- `::numeric` is added before `ROUND` where the value is a double precision number.

## Team Members

1. 202618016 - Krishika Lalwani
2. 202618013 - Devanshi vora
3. 202618052 - Lakshita Pagaria

In [1]:
import pandas as pd
from sqlalchemy import create_engine, URL, text

# Show all columns of a result on the console
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

## Create the SQLAlchemy connection

In [2]:
username = "202618052"
password = "Ljp_@2006#"
host = "10.100.71.21"
port = 5432
database = "202618052"
schema = "DBMS_project"

url = URL.create(
    "postgresql+psycopg",
    username=username,
    password=password,
    host=host,
    port=port,
    database=database
)

# Create the engine (schema name in quotes so its capital letters are kept)
engine = create_engine(
    url,
    connect_args={
        "options": f'-csearch_path="{schema}",public'
    }
)

print(f"Connected to PostgreSQL database '{database}'")
print(f"Using schema '{schema}'")

Connected to PostgreSQL database '202618052'
Using schema 'DBMS_project'


In [3]:
try:
    with engine.connect() as connection:
        print("PostgreSQL connection successful!")
        print(connection.execute(text("SELECT version();")).scalar())
        print("Current schema:", connection.execute(text("SELECT current_schema();")).scalar())
except Exception as e:
    print("Connection failed:")
    print(e)

PostgreSQL connection successful!
PostgreSQL 10.20 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 4.8.5 20150623 (Red Hat 4.8.5-44), 64-bit
Current schema: DBMS_project


## check the tables before running the queries

Number of rows in each of the 12 tables of the `DBMS_project` schema.

In [4]:
table_list = [
    "circuits", "constructors", "drivers", "status", "races", "qualifying",
    "results", "sprint_results", "lap_times", "pit_stops",
    "driver_standings", "constructor_standings"
]

with engine.connect() as connection:
    for table_name in table_list:
        query = text(f'SELECT COUNT(*) FROM "{schema}"."{table_name}"')
        row_count = connection.execute(query).scalar()
        print(f"Table: {table_name} | Total rows: {row_count}")

Table: circuits | Total rows: 77
Table: constructors | Total rows: 212
Table: drivers | Total rows: 861
Table: status | Total rows: 139
Table: races | Total rows: 1125
Table: qualifying | Total rows: 10494
Table: results | Total rows: 26759
Table: sprint_results | Total rows: 360


Table: lap_times | Total rows: 589081
Table: pit_stops | Total rows: 11371
Table: driver_standings | Total rows: 34863
Table: constructor_standings | Total rows: 13391


## function to run a query

Each query is submitted with `pd.read_sql_query`, the result is collected into a DataFrame,
and `head(10)` of the DataFrame is printed on the console.

In [5]:
def run_query(sql_query):
    df = pd.read_sql_query(text(sql_query), engine)
    print(f"Rows returned: {len(df)}\n")
    print(df.head(10))
    return df

## function to create a view

`create_view` runs the `CREATE OR REPLACE VIEW` statement on the server, then reads the view back with
`read_sql_query` and prints `head(10)`.

In [6]:
def create_view(view_sql, view_name):
    with engine.begin() as connection:
        connection.execute(text(view_sql))
    print(f"View {view_name} created.\n")
    return run_query(f'SELECT * FROM "{schema}".{view_name}')

## View 1: v_driver_career_wins

**Based on:** Query 1 – Career wins
**Purpose:** Career wins, podiums and win percentage of every driver

In [7]:
view_1 = r"""
CREATE OR REPLACE VIEW "DBMS_project".v_driver_career_wins AS
SELECT d.forename || ' ' || d.surname                         AS driver,
       d.nationality,
       COUNT(*) FILTER (WHERE NULLIF(r.position, '\N')::int = 1)                 AS wins,
       COUNT(*) FILTER (WHERE NULLIF(r.position, '\N')::int <= 3)                AS podiums,
       COUNT(*)                                               AS races_entered,
       ROUND(100.0 * COUNT(*) FILTER (WHERE NULLIF(r.position, '\N')::int = 1) / COUNT(*), 1) AS win_pct
FROM results r
JOIN drivers d ON d."driverId" = r."driverId"
GROUP BY d."driverId", d.forename, d.surname, d.nationality
ORDER BY wins DESC, podiums DESC;
"""

view_1_df = create_view(view_1, "v_driver_career_wins")

View v_driver_career_wins created.

Rows returned: 861

               driver nationality  wins  podiums  races_entered  win_pct
0      Lewis Hamilton     British   105      202            356     29.5
1  Michael Schumacher      German    91      155            308     29.5
2      Max Verstappen       Dutch    63      112            209     30.1
3    Sebastian Vettel      German    53      122            300     17.7
4         Alain Prost      French    51      106            202     25.2
5        Ayrton Senna   Brazilian    41       80            162     25.3
6     Fernando Alonso     Spanish    32      106            404      7.9
7       Nigel Mansell     British    31       59            192     16.1
8      Jackie Stewart     British    27       43            100     27.0
9          Niki Lauda    Austrian    25       54            174     14.4


## View 2: v_constructor_championships

**Based on:** Query 2 – Constructor dominance
**Purpose:** Constructors' Championship titles and title years of every constructor

In [8]:
view_2 = r"""
CREATE OR REPLACE VIEW "DBMS_project".v_constructor_championships AS
WITH season_final_race AS (          -- last race of every season
    SELECT year, MAX(round) AS last_round
    FROM races
    GROUP BY year
),
champions AS (                       -- constructor in P1 after the final race
    SELECT ra.year, cs."constructorId"
    FROM constructor_standings cs
    JOIN races ra             ON ra."raceId" = cs."raceId"
    JOIN season_final_race sf ON sf.year = ra.year AND sf.last_round = ra.round
    WHERE cs.position = 1
)
SELECT c.name                                             AS constructor,
       COUNT(*)                                           AS championships,
       MIN(ch.year)                                       AS first_title,
       MAX(ch.year)                                       AS latest_title,
       STRING_AGG(ch.year::text, ', ' ORDER BY ch.year)   AS title_years
FROM champions ch
JOIN constructors c ON c."constructorId" = ch."constructorId"
GROUP BY c.name
ORDER BY championships DESC, first_title;
"""

view_2_df = create_view(view_2, "v_constructor_championships")

View v_constructor_championships created.

Rows returned: 17

     constructor  championships  first_title  latest_title                                                                      title_years
0        Ferrari             16         1961          2008  1961, 1964, 1975, 1976, 1977, 1979, 1982, 1983, 1999, 2000, 2001, 2002, 2003...
1        McLaren              9         1974          2024                             1974, 1984, 1985, 1988, 1989, 1990, 1991, 1998, 2024
2       Williams              9         1980          1997                             1980, 1981, 1986, 1987, 1992, 1993, 1994, 1996, 1997
3       Mercedes              8         2014          2021                                   2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021
4       Red Bull              6         2010          2023                                               2010, 2011, 2012, 2013, 2022, 2023
5     Team Lotus              4         1970          1978                                        

## View 3: v_driver_career_points

**Based on:** Query 3 – All-time points leaders
**Purpose:** Career championship points, seasons and titles of every driver

In [9]:
view_3 = r"""
CREATE OR REPLACE VIEW "DBMS_project".v_driver_career_points AS
WITH season_final_race AS (
    SELECT year, MAX(round) AS last_round
    FROM races
    GROUP BY year
),
season_end AS (                      -- each driver's final standing in each season
    SELECT ds."driverId", ra.year, ds.points, ds.position
    FROM driver_standings ds
    JOIN races ra             ON ra."raceId" = ds."raceId"
    JOIN season_final_race sf ON sf.year = ra.year AND sf.last_round = ra.round
)
SELECT d.forename || ' ' || d.surname            AS driver,
       SUM(se.points)                            AS career_points,
       COUNT(*)                                  AS seasons,
       COUNT(*) FILTER (WHERE se.position = 1)   AS world_titles,
       MAX(se.points)                            AS best_season_points,
       ROUND((SUM(se.points) / COUNT(*))::numeric, 1)       AS points_per_season
FROM season_end se
JOIN drivers d ON d."driverId" = se."driverId"
GROUP BY d."driverId", d.forename, d.surname
ORDER BY career_points DESC;
"""

view_3_df = create_view(view_3, "v_driver_career_points")

View v_driver_career_points created.

Rows returned: 854

               driver  career_points  seasons  world_titles  best_season_points  points_per_season
0      Lewis Hamilton         4862.5       18             7               413.0              270.1
1    Sebastian Vettel         3098.0       16             4               397.0              193.6
2      Max Verstappen         3023.5       10             4               575.0              302.4
3     Fernando Alonso         2337.0       21             2               278.0              111.3
4      Kimi Räikkönen         1873.0       19             1               251.0               98.6
5     Valtteri Bottas         1797.0       12             0               326.0              149.8
6        Sergio Pérez         1638.0       14             0               305.0              117.0
7        Nico Rosberg         1594.5       11             1               385.0              145.0
8  Michael Schumacher         1566.0       19      

## View 4: v_pitstop_efficiency

**Based on:** Query 4 – Pit-stop efficiency
**Purpose:** Pit-stop performance of each constructor across seasons

In [10]:
view_4 = r"""
CREATE OR REPLACE VIEW "DBMS_project".v_pitstop_efficiency AS
WITH stops AS (                      -- attach the constructor and season to every stop
    SELECT ra.year, r."constructorId", ps.milliseconds
    FROM pit_stops ps
    JOIN results r ON r."raceId" = ps."raceId" AND r."driverId" = ps."driverId"
    JOIN races ra  ON ra."raceId" = ps."raceId"
    WHERE ps.milliseconds < 60000    -- exclude red-flag stoppages and long repairs
),
season_avg AS (
    SELECT year, "constructorId",
           COUNT(*)                         AS stops,
           AVG(milliseconds) / 1000.0       AS avg_stop_s,
           RANK() OVER (PARTITION BY year ORDER BY AVG(milliseconds)) AS season_rank
    FROM stops
    GROUP BY year, "constructorId"
    HAVING COUNT(*) >= 20            -- ignore teams with very few stops in a season
)
SELECT c.name                                       AS constructor,
       COUNT(*)                                     AS seasons,
       COUNT(*) FILTER (WHERE sa.season_rank = 1)   AS seasons_fastest,
       ROUND(AVG(sa.season_rank), 1)                AS avg_season_rank,
       ROUND(MIN(sa.avg_stop_s), 2)                 AS best_season_avg_s,
       ROUND(AVG(sa.avg_stop_s), 2)                 AS overall_avg_s
FROM season_avg sa
JOIN constructors c ON c."constructorId" = sa."constructorId"
GROUP BY c.name
HAVING COUNT(*) >= 3
ORDER BY seasons_fastest DESC, avg_season_rank;
"""

view_4_df = create_view(view_4, "v_pitstop_efficiency")

View v_pitstop_efficiency created.

Rows returned: 17

      constructor  seasons  seasons_fastest  avg_season_rank  best_season_avg_s  overall_avg_s
0        Red Bull       14                7              2.3              22.15          23.75
1        Mercedes       14                4              2.5              22.46          23.77
2         Ferrari       14                2              3.4              22.24          23.97
3         McLaren       14                1              5.0              22.58          24.31
4    Aston Martin        4                0              4.8              24.13          24.55
5  Alpine F1 Team        4                0              4.8              24.00          24.78
6         Renault        6                0              5.3              23.24          24.37
7     Force India        8                0              6.0              22.69          24.16
8      AlphaTauri        4                0              6.0              24.54          2

## View 5: v_dnf_analysis

**Based on:** Query 9 – DNF / retirement analysis
**Purpose:** Race retirements of each constructor grouped by cause

In [11]:
view_5 = r"""
CREATE OR REPLACE VIEW "DBMS_project".v_dnf_analysis AS
WITH dnf AS (
    SELECT r."constructorId",
           s.status,
           CASE
             WHEN s.status ILIKE ANY (ARRAY['%engine%', '%power unit%', '%turbo%', '%overheating%', '%oil%',
                                            '%water%', '%radiator%', '%ers%', '%fuel%', '%ignition%',
                                            '%injection%', '%exhaust%', '%alternator%', '%magneto%', '%throttle%'])
                  THEN 'Engine / power unit'
             WHEN s.status ILIKE ANY (ARRAY['%accident%', '%collision%', '%spun off%', '%damage%'])
                  THEN 'Accident / collision'
             WHEN s.status ILIKE ANY (ARRAY['%gearbox%', '%transmission%', '%clutch%', '%driveshaft%',
                                            '%halfshaft%', '%differential%', '%axle%'])
                  THEN 'Gearbox / transmission'
             ELSE 'Other'
           END AS cause_group
    FROM results r
    JOIN status s ON s."statusId" = r."statusId"
    WHERE s.status <> 'Finished'
      AND s.status NOT LIKE '+%'                     -- '+1 Lap' etc. are classified finishers
      AND s.status NOT IN ('Did not qualify', 'Did not prequalify', 'Withdrew',
                           'Did not start', 'Disqualified', 'Excluded', '107% Rule')
)
SELECT c.name                                                          AS constructor,
       COUNT(*)                                                        AS dnfs,
       COUNT(*) FILTER (WHERE cause_group = 'Engine / power unit')     AS engine,
       COUNT(*) FILTER (WHERE cause_group = 'Accident / collision')    AS accident,
       COUNT(*) FILTER (WHERE cause_group = 'Gearbox / transmission')  AS gearbox,
       COUNT(*) FILTER (WHERE cause_group = 'Other')                   AS other,
       MODE() WITHIN GROUP (ORDER BY status)                           AS most_common_cause
FROM dnf
JOIN constructors c ON c."constructorId" = dnf."constructorId"
GROUP BY c.name
ORDER BY dnfs DESC;
"""

view_5_df = create_view(view_5, "v_dnf_analysis")

View v_dnf_analysis created.



Rows returned: 185

  constructor  dnfs  engine  accident  gearbox  other most_common_cause
0     Ferrari   708     279       203      102    124            Engine
1     McLaren   552     193       188       62    109            Engine
2    Williams   486     137       184       56    109            Engine
3  Team Lotus   416     152       108       76     80            Engine
4     Tyrrell   382     136       137       60     49            Engine
5     Minardi   351      97       109       82     63            Engine
6     Brabham   331     145        68       53     65            Engine
7      Ligier   285     107        91       36     51            Engine
8      Arrows   282     100        79       39     64            Engine
9         BRM   271     146        43       44     38            Engine


## View 6: v_season_points_progression_2021

**Based on:** Query 10 – Season points progression
**Purpose:** Race-by-race championship progression of the 2021 top 5

In [12]:
view_6 = r"""
CREATE OR REPLACE VIEW "DBMS_project".v_season_points_progression_2021 AS
WITH season_races AS (
    SELECT "raceId", round, name
    FROM races
    WHERE year = 2021
),
top5 AS (                            -- top 5 of the final standings
    SELECT ds."driverId"
    FROM driver_standings ds
    WHERE ds."raceId" = (SELECT "raceId" FROM season_races ORDER BY round DESC LIMIT 1)
      AND ds.position <= 5
)
SELECT sr.round,
       sr.name                                                        AS race,
       d.surname                                                      AS driver,
       ds.points                                                      AS total_points,
       ds.points - COALESCE(LAG(ds.points, 1) OVER (PARTITION BY ds."driverId" ORDER BY sr.round), 0) AS points_this_race,
       MAX(ds.points) OVER (PARTITION BY sr.round) - ds.points        AS gap_to_leader,
       ds.position                                                    AS championship_position
FROM driver_standings ds
JOIN season_races sr ON sr."raceId" = ds."raceId"
JOIN top5 t          ON t."driverId" = ds."driverId"
JOIN drivers d       ON d."driverId" = ds."driverId"
ORDER BY sr.round, ds.position;
"""

view_6_df = create_view(view_6, "v_season_points_progression_2021")

View v_season_points_progression_2021 created.

Rows returned: 110

   round                       race      driver  total_points  points_this_race  gap_to_leader  championship_position
0      1         Bahrain Grand Prix    Hamilton          25.0              25.0            0.0                      1
1      1         Bahrain Grand Prix  Verstappen          18.0              18.0            7.0                      2
2      1         Bahrain Grand Prix      Bottas          16.0              16.0            9.0                      3
3      1         Bahrain Grand Prix       Pérez          10.0              10.0           15.0                      5
4      1         Bahrain Grand Prix       Sainz           4.0               4.0           21.0                      8
5      2  Emilia Romagna Grand Prix    Hamilton          44.0              19.0            0.0                      1
6      2  Emilia Romagna Grand Prix  Verstappen          43.0              25.0            1.0            

## View 7: v_home_race_advantage

**Based on:** Query 12 – Home-race advantage
**Purpose:** Home Grand Prix finishing position compared with the driver's other races

In [13]:
view_7 = r"""
CREATE OR REPLACE VIEW "DBMS_project".v_home_race_advantage AS
WITH nationality_country(nationality, country) AS (
    VALUES ('British','UK'), ('American','USA'), ('American','United States'), ('Italian','Italy'),
           ('French','France'), ('German','Germany'), ('Brazilian','Brazil'), ('Argentine','Argentina'),
           ('Belgian','Belgium'), ('South African','South Africa'), ('Swiss','Switzerland'),
           ('Japanese','Japan'), ('Australian','Australia'), ('Dutch','Netherlands'), ('Spanish','Spain'),
           ('Austrian','Austria'), ('Canadian','Canada'), ('Swedish','Sweden'), ('Mexican','Mexico'),
           ('Monegasque','Monaco'), ('Portuguese','Portugal'), ('Russian','Russia'), ('Chinese','China'),
           ('Indian','India'), ('Malaysian','Malaysia'), ('Hungarian','Hungary')
),
entries AS (
    SELECT r."driverId", ra.year, r."positionOrder",
           CASE WHEN nc.country IS NOT NULL THEN 1 ELSE 0 END AS is_home
    FROM results r
    JOIN races ra    ON ra."raceId" = r."raceId"
    JOIN circuits ci ON ci."circuitId" = ra."circuitId"
    JOIN drivers d   ON d."driverId" = r."driverId"
    LEFT JOIN nationality_country nc ON nc.nationality = d.nationality AND nc.country = ci.country
),
per_season AS (                      -- home races vs. the driver's other races in the same season
    SELECT "driverId", year,
           COUNT(*) FILTER (WHERE is_home = 1)                AS home_races,
           AVG("positionOrder") FILTER (WHERE is_home = 1)    AS avg_home,
           AVG("positionOrder") FILTER (WHERE is_home = 0)    AS avg_other
    FROM entries
    GROUP BY "driverId", year
    HAVING COUNT(*) FILTER (WHERE is_home = 1) > 0
       AND COUNT(*) FILTER (WHERE is_home = 0) > 0
)
SELECT d.forename || ' ' || d.surname                AS driver,
       d.nationality,
       SUM(ps.home_races)::int                       AS home_races,
       ROUND(AVG(ps.avg_home), 2)                    AS avg_finish_home,
       ROUND(AVG(ps.avg_other), 2)                   AS avg_finish_other,
       ROUND(AVG(ps.avg_other - ps.avg_home), 2)     AS places_gained_at_home,
       CASE WHEN AVG(ps.avg_other - ps.avg_home) > 0 THEN 'Better at home'
            ELSE 'Not better at home' END            AS verdict
FROM per_season ps
JOIN drivers d ON d."driverId" = ps."driverId"
GROUP BY d."driverId", d.forename, d.surname, d.nationality
HAVING SUM(ps.home_races) >= 8
ORDER BY home_races DESC;
"""

view_7_df = create_view(view_7, "v_home_race_advantage")

View v_home_race_advantage created.

Rows returned: 71

                 driver nationality  home_races  avg_finish_home  avg_finish_other  places_gained_at_home             verdict
0      Riccardo Patrese     Italian          29            13.91             12.66                  -1.25  Not better at home
1      Michele Alboreto     Italian          28            15.46             14.41                  -1.06  Not better at home
2    Michael Schumacher      German          28             5.35              6.72                   1.37      Better at home
3     Andrea de Cesaris     Italian          27            14.96             16.14                   1.18      Better at home
4       Fernando Alonso     Spanish          26             8.05              8.48                   0.43      Better at home
5          Jarno Trulli     Italian          25            13.43             12.17                  -1.26  Not better at home
6  Giancarlo Fisichella     Italian          24            11.

## View 8: v_decade_team_dominance

**Based on:** Query 13 – Decade-wise team dominance
**Purpose:** Top 3 constructors of each decade by share of points

In [14]:
view_8 = r"""
CREATE OR REPLACE VIEW "DBMS_project".v_decade_team_dominance AS
WITH season_final_race AS (
    SELECT year, MAX(round) AS last_round
    FROM races
    GROUP BY year
),
season_points AS (                   -- constructors' final points of each season
    SELECT ra.year, cs."constructorId", cs.points
    FROM constructor_standings cs
    JOIN races ra             ON ra."raceId" = cs."raceId"
    JOIN season_final_race sf ON sf.year = ra.year AND sf.last_round = ra.round
),
decade_points AS (
    SELECT (year / 10) * 10 AS decade, "constructorId", SUM(points) AS points
    FROM season_points
    GROUP BY decade, "constructorId"
),
ranked AS (
    SELECT decade, "constructorId", points,
           ROUND((100.0 * points / SUM(points) OVER (PARTITION BY decade))::numeric, 1) AS pct_share,
           RANK() OVER (PARTITION BY decade ORDER BY points DESC)           AS rank_in_decade
    FROM decade_points
)
SELECT r.decade, r.rank_in_decade, c.name AS constructor, r.points, r.pct_share
FROM ranked r
JOIN constructors c ON c."constructorId" = r."constructorId"
WHERE r.rank_in_decade <= 3
ORDER BY r.decade, r.rank_in_decade;
"""

view_8_df = create_view(view_8, "v_decade_team_dominance")

View v_decade_team_dominance created.

Rows returned: 24

   decade  rank_in_decade    constructor  points  pct_share
0    1950               1        Ferrari    72.0       29.9
1    1950               2        Vanwall    48.0       19.9
2    1950               3  Cooper-Climax    40.0       16.6
3    1960               1        Ferrari   276.0       15.1
4    1960               2            BRM   254.0       13.9
5    1960               3   Lotus-Climax   224.0       12.3
6    1970               1        Ferrari   616.5       19.7
7    1970               2     Team Lotus   479.0       15.3
8    1970               3        Tyrrell   447.0       14.3
9    1980               1        McLaren   887.5       22.9


## View 9: v_grid_vs_points

**Based on:** Query 14 – Grid position vs. points scored
**Purpose:** Points, wins and podiums by starting grid position (2010 onward)

In [15]:
view_9 = r"""
CREATE OR REPLACE VIEW "DBMS_project".v_grid_vs_points AS
SELECT CASE
         WHEN r.grid = 0               THEN 'Pit lane'
         WHEN r.grid = 1               THEN 'P1 (pole)'
         WHEN r.grid BETWEEN 2 AND 3   THEN 'P2-P3'
         WHEN r.grid BETWEEN 4 AND 6   THEN 'P4-P6'
         WHEN r.grid BETWEEN 7 AND 10  THEN 'P7-P10'
         WHEN r.grid BETWEEN 11 AND 15 THEN 'P11-P15'
         ELSE 'P16+'
       END                                                                    AS grid_bucket,
       COUNT(*)                                                               AS starts,
       ROUND(AVG(r.points)::numeric, 2)                                                AS avg_points,
       ROUND(100.0 * COUNT(*) FILTER (WHERE NULLIF(r.position, '\N')::int = 1)  / COUNT(*), 1)   AS win_pct,
       ROUND(100.0 * COUNT(*) FILTER (WHERE NULLIF(r.position, '\N')::int <= 3) / COUNT(*), 1)   AS podium_pct,
       ROUND(100.0 * COUNT(*) FILTER (WHERE r.points > 0)    / COUNT(*), 1)   AS points_finish_pct
FROM results r
JOIN races ra ON ra."raceId" = r."raceId"
WHERE ra.year >= 2010
GROUP BY grid_bucket
ORDER BY MIN(CASE WHEN r.grid = 0 THEN 99 ELSE r.grid END);
"""

view_9_df = create_view(view_9, "v_grid_vs_points")

View v_grid_vs_points created.



Rows returned: 7

  grid_bucket  starts  avg_points  win_pct  podium_pct  points_finish_pct
0   P1 (pole)     305       18.70     51.1        80.0               90.5
1       P2-P3     608       14.06     18.3        59.9               88.0
2       P4-P6     915        8.81      2.6        23.0               80.1
3      P7-P10    1216        4.13      0.8         4.9               62.5
4     P11-P15    1518        1.65      0.2         1.6               33.7
5        P16+    1789        0.60      0.1         0.8               12.3
6    Pit lane      85        0.61      0.0         0.0               16.5


## View 10: v_sprint_vs_gp

**Based on:** Query 15 – Sprint vs. Grand Prix performance
**Purpose:** Sprint result compared with the Grand Prix result on the same weekend

In [16]:
view_10 = r"""
CREATE OR REPLACE VIEW "DBMS_project".v_sprint_vs_gp AS
SELECT d.forename || ' ' || d.surname                              AS driver,
       COUNT(*)                                                    AS sprint_weekends,
       ROUND(AVG(s."positionOrder"), 2)                            AS avg_sprint_finish,
       ROUND(AVG(r."positionOrder"), 2)                            AS avg_gp_finish,
       ROUND(AVG(r."positionOrder" - s."positionOrder"), 2)        AS gp_minus_sprint,
       SUM(s.points)                                               AS sprint_points,
       SUM(r.points)                                               AS gp_points_same_weekends,
       COUNT(*) FILTER (WHERE s."positionOrder" = r."positionOrder") AS same_position_both
FROM sprint_results s
JOIN results r ON r."raceId" = s."raceId" AND r."driverId" = s."driverId"
JOIN races ra  ON ra."raceId" = s."raceId"
JOIN drivers d ON d."driverId" = s."driverId"
GROUP BY d."driverId", d.forename, d.surname
HAVING COUNT(*) >= 5
ORDER BY sprint_points DESC;
"""

view_10_df = create_view(view_10, "v_sprint_vs_gp")

View v_sprint_vs_gp created.

Rows returned: 23

            driver  sprint_weekends  avg_sprint_finish  avg_gp_finish  gp_minus_sprint  sprint_points  gp_points_same_weekends  same_position_both
0   Max Verstappen               18               2.00           3.83             1.83          111.0                    335.0                   7
1     Carlos Sainz               18               4.89           8.61             3.72           69.0                    141.0                   4
2  Charles Leclerc               18               5.06           5.61             0.56           67.0                    223.0                   1
3     Lando Norris               18               6.33           6.33             0.00           57.0                    189.0                   2
4     Sergio Pérez               18               8.06           7.06            -1.00           53.0                    169.0                   3
5   George Russell               18               7.28           6.50

## check the views created

Lists all the views saved in the `DBMS_project` schema.

In [17]:
views_df = pd.read_sql_query(text("""
    SELECT table_schema, table_name AS view_name
    FROM information_schema.views
    WHERE table_schema = 'DBMS_project'
    ORDER BY table_name;
"""), engine)
print(views_df)

   table_schema                         view_name
0  DBMS_project       v_constructor_championships
1  DBMS_project           v_decade_team_dominance
2  DBMS_project                    v_dnf_analysis
3  DBMS_project            v_driver_career_points
4  DBMS_project              v_driver_career_wins
5  DBMS_project                  v_grid_vs_points
6  DBMS_project             v_home_race_advantage
7  DBMS_project              v_pitstop_efficiency
8  DBMS_project  v_season_points_progression_2021
9  DBMS_project                    v_sprint_vs_gp


In [18]:
engine.dispose()
print("Database connection closed.")

Database connection closed.
